In [ ]:
%%writefile fmm.cpp
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;
typedef std::pair<float, int> Entry; // coppia (u, indice) per la coda di priorita'

float solve_quadratic(float a, float b, float F, float h)
{
    float Fh = F * h;
    if (fabsf(a - b) <= Fh)                                   
        return 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));
    return fminf(a, b) + Fh;                                  
}

float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int i, int j, int N, float h)
{
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[i * N + (j - 1)] : INF;
    float right = (j < N - 1) ? u[i * N + (j + 1)] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[(i - 1) * N + j] : INF;
    float up    = (i < N - 1) ? u[(i + 1) * N + j] : INF;
    // Per arrivare devi scorrere come un indice globale: u[i][j] = u[i * N + j]!

    float a = fminf(left, right);
    float b = fminf(down, up);

    int idx = i * N + j;
    float q = solve_quadratic(a, b, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ij, u)
}
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, int src)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N, LONTANO);    // array di char NxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    heap.push(Entry(u[src], src));                   // la sorgente e' il primo Trial
    state[src] = ATTIVO;

    const int di[4] = {  0, 0, -1, 1 };              // spostamenti verso i 4 vicini:
    const int dj[4] = { -1, 1,  0, 0 };              // sinistra, destra, sotto, sopra

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int i = x / N, j = x % N;                    // riga e colonna del Trial
        for (int k = 0; k < 4; ++k) {                // foreach x_nb adiacente a Trial
            int ni = i + di[k], nj = j + dj[k];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N) continue;   // fuori dal dominio
            int nb = ni * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h;   // coordinate del nodo (i, j)
            double U = std::sqrt(x * x + y * y);        // soluzione esatta
            double d = u[i * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
        }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}
int main()
{   
    // Test sorgente puntiforme in 2D, con F = 1.0 e dominio [-1, 1]^2.
    const int sizes[] = { 101, 401, 1001, 2001 };
    double e_prec = 0.0, h_prec = 0.0;
    printf("    N        h     errore rel.   ordine   tempo FMM [ms]\n");
    // Loop sulle dimensioni della griglia
    for (int N : sizes) {
    const float h = 2.0f / (N - 1);         // spaziatura del grigliato
    std::vector<float> u(N * N, INF);       // array di float NxN, inizializzato a INF
    std::vector<float> F(N * N, 1.0f);      // array di float NxN, inizializzato a 1.0f

    // Sorgente puntiforme nell'origine
    const int si = N / 2, sj = N / 2;       // riga e colonna del nodo centrale
    u[si * N + sj] = 0.0f;  // equivalente u[si][sj] = 0.0f
    
    // Fast Marching, con misura del tempo
        auto t0 = std::chrono::high_resolution_clock::now(); // inizio misurazione del tempo
        fmm(u, F, N, h, si * N + sj);
        auto t1 = std::chrono::high_resolution_clock::now(); // fine misurazione del tempo
        double ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

        // Errore relativo (eq. 4.2) e ordine di convergenza stimato
        double e = relative_error(u, N, h);
        if (e_prec > 0.0)
            printf("%5d  %.5f   %.3e   %6.2f   %12.1f\n", N, h, e, std::log(e_prec / e) / std::log(h_prec / h), ms);
        else
            printf("%5d  %.5f   %.3e      -     %12.1f\n", N, h, e, ms);
        e_prec = e;
        h_prec = h;

        // Scrittura su file dei risultati per la visualizzazione con Python
        if (N == 101) {
            std::ofstream out("fmm_101.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(u.data()), u.size() * sizeof(float));
        }
    }
    return 0;
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Lettura della soluzione salvata dal programma C++ ----
N = 101
u = np.fromfile("fmm_101.bin", dtype=np.float32).reshape(N, N)   # u[i, j]: riga i = y, colonna j = x

# ---- Soluzione esatta (eq. 4.3) sulla stessa griglia ----
x = np.linspace(-1.0, 1.0, N)
X, Y = np.meshgrid(x, x)
U = np.sqrt(X**2 + Y**2)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Sinistra: curve di livello di u (FMM) e di U (esatta), come nella fig. 4.1 della tesi
livelli = np.arange(0.2, 1.5, 0.2)
c = ax[0].contour(X, Y, u, levels=livelli, colors="tab:blue")
ax[0].contour(X, Y, U, levels=livelli, colors="k", linestyles="--")
ax[0].clabel(c, fmt="%.1f", fontsize=8)
ax[0].plot(0, 0, "ro")                                          # sorgente
ax[0].set_aspect("equal")
ax[0].set_title("Fast Marching (blu) e soluzione esatta (tratteggio)")

# Destra: errore |u - U| in ogni nodo
im = ax[1].imshow(np.abs(u - U), origin="lower", extent=[-1, 1, -1, 1], cmap="viridis")
fig.colorbar(im, ax=ax[1], label="|u - U|")
ax[1].set_title("Errore assoluto")

for a in ax:
    a.set_xlabel("x [u.a.]")
    a.set_ylabel("y [u.a.]")
plt.tight_layout()
plt.show()